# PG-Stack · Step 4b — PG-Boost: graph-neighbourhood features + LightGBM

**Why:** Step 4 showed a GNN learning from raw features loses to LightGBM. So here the **graph builds the features** and **LightGBM learns**.

**New idea — local vs global ugly duckling**
- *Global* (= `tab_pnorm`): lesion vs **whole patient** (z-score).
- *Local* (new, graph): lesion vs its **10 most similar-looking lesions** (DINOv2 kNN) and its **10 nearest lesions on the body** (3D kNN) + local outlier scores.

| Feature set | Content |
|---|---|
| `G0_lesion` | lesion only (reproduces tab_lesion) |
| `G1_global` | + global patient z-scores (reproduces tab_pnorm) |
| `G2_local` | + **local graph** features only |
| `G3_global_local` | + both → **PG-Boost** |
| `pgboost_stack` | LR stack of G3 + emb_lr + cnn → final system |

Same folds, same LightGBM recipe as Step 1. **CPU is enough** (~20–30 min). Same session: reads/writes `/kaggle/working`.

In [ ]:
import os, sys, time, gc, warnings
import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings("ignore")

WORK = "/kaggle/working"
COMP_DIR = "/kaggle/input/competitions/isic-2024-challenge"
sys.path.insert(0, WORK)
from pgstack_eval import *

SEED, NEG_SAMPLE_RATIO, N_BAGS = 42, 0.01, 5      # identical to Step 1
K_APP, K_POS = 10, 10
for f in ["oof_step3.csv", "emb_pca32.npy", "pgstack_eval.py"]:
    assert os.path.exists(f"{WORK}/{f}"), f"{f} missing"
assert os.path.exists(f"{COMP_DIR}/train-metadata.csv"), "check COMP_DIR"

## 1. Rebuild Step-1 lesion features (same code) and align to Step-3 row order

In [ ]:
oof = pd.read_csv(f"{WORK}/oof_step3.csv")
df = pd.read_csv(f"{COMP_DIR}/train-metadata.csv", low_memory=False)
df = oof[["isic_id", "fold"]].merge(df, on="isic_id", how="left")
assert len(df) == len(oof) and (df.isic_id.values == oof.isic_id.values).all()
emb = np.load(f"{WORK}/emb_pca32.npy").astype(np.float32)

LEAK = ["lesion_id", "iddx_full", "iddx_1", "iddx_2", "iddx_3", "iddx_4", "iddx_5",
        "mel_mitotic_index", "mel_thick_mm", "tbp_lv_dnn_lesion_confidence"]
NUM_RAW = [c for c in df.columns if (c.startswith("tbp_lv_") or c in ["age_approx", "clin_size_long_diam_mm"])
           and c not in LEAK and pd.api.types.is_numeric_dtype(df[c])]
CAT_COLS = ["sex", "anatom_site_general", "tbp_tile_type", "tbp_lv_location", "tbp_lv_location_simple"]

def add_lesion_features(d):
    d = d.copy(); eps = 1e-6
    d["age_approx"] = pd.to_numeric(d["age_approx"], errors="coerce")
    d["lesion_size_ratio"] = d.tbp_lv_minorAxisMM / (d.clin_size_long_diam_mm + eps)
    d["lesion_shape_index"] = d.tbp_lv_areaMM2 / (d.tbp_lv_perimeterMM ** 2 + eps)
    d["hue_contrast"] = (d.tbp_lv_H - d.tbp_lv_Hext).abs()
    d["luminance_contrast"] = (d.tbp_lv_L - d.tbp_lv_Lext).abs()
    d["lesion_color_difference"] = np.sqrt(d.tbp_lv_deltaA**2 + d.tbp_lv_deltaB**2 + d.tbp_lv_deltaL**2)
    d["border_complexity"] = d.tbp_lv_norm_border + d.tbp_lv_symm_2axis
    d["color_uniformity"] = d.tbp_lv_color_std_mean / (d.tbp_lv_radial_color_std_max + eps)
    d["position_distance_3d"] = np.sqrt(d.tbp_lv_x**2 + d.tbp_lv_y**2 + d.tbp_lv_z**2)
    d["perimeter_to_area_ratio"] = d.tbp_lv_perimeterMM / (d.tbp_lv_areaMM2 + eps)
    d["area_to_perimeter_ratio"] = d.tbp_lv_areaMM2 / (d.tbp_lv_perimeterMM + eps)
    d["lesion_visibility_score"] = d.tbp_lv_deltaLBnorm + d.tbp_lv_norm_color
    d["symmetry_border_consistency"] = d.tbp_lv_symm_2axis * d.tbp_lv_norm_border
    d["color_shape_composite"] = (d.tbp_lv_color_std_mean + d.tbp_lv_area_perim_ratio + d.tbp_lv_symm_2axis) / 3
    d["log_area"] = np.log1p(d.tbp_lv_areaMM2)
    d["normalized_lesion_size"] = d.clin_size_long_diam_mm / (d.age_approx + eps)
    d["mean_hue_difference"] = (d.tbp_lv_H + d.tbp_lv_Hext) / 2
    d["std_dev_contrast"] = np.sqrt((d.tbp_lv_deltaA**2 + d.tbp_lv_deltaB**2 + d.tbp_lv_deltaL**2) / 3)
    return d

ENG = ["lesion_size_ratio", "lesion_shape_index", "hue_contrast", "luminance_contrast", "lesion_color_difference",
       "border_complexity", "color_uniformity", "position_distance_3d", "perimeter_to_area_ratio",
       "area_to_perimeter_ratio", "lesion_visibility_score", "symmetry_border_consistency",
       "color_shape_composite", "log_area", "normalized_lesion_size", "mean_hue_difference", "std_dev_contrast"]
df = add_lesion_features(df)
LESION_NUM = sorted(set(NUM_RAW + ENG))
for c in CAT_COLS: df[c] = df[c].fillna("NA").astype("category")
y, folds, pid = oof.target.values, oof.fold.values, oof.patient_id.values
print(df.shape, "| lesion numeric features:", len(LESION_NUM))

## 2. Global patient context (exactly as Step 1)

In [ ]:
g = df.groupby("patient_id")
GLOBAL = []
for c in LESION_NUM:
    df[f"{c}_pnorm"] = (df[c] - g[c].transform("mean")) / (g[c].transform("std") + 1e-6)
    GLOBAL.append(f"{c}_pnorm")
df["patient_n_lesions"] = g["isic_id"].transform("size"); GLOBAL.append("patient_n_lesions")

## 3. Local graph context (NEW)
Per patient: kNN graph in **appearance** space (DINOv2) and in **3D body** space.
For every lesion feature: `lesion − mean(neighbours)` scaled by the neighbours' spread, plus **local outlier scores** (how far the lesion is from its neighbours, relative to the patient's typical distance).

In [ ]:
X = df[LESION_NUM].astype(np.float32)
X = X.fillna(X.median()).values
XYZ = df[["tbp_lv_x", "tbp_lv_y", "tbp_lv_z"]].values.astype(np.float32)

loc_app = np.zeros_like(X); loc_pos = np.zeros_like(X)
out_app = np.zeros(len(df), np.float32); out_pos = np.zeros(len(df), np.float32)
out_app_rel = np.zeros(len(df), np.float32); out_pos_rel = np.zeros(len(df), np.float32)

def neigh(Z, k):
    n = len(Z); kk = min(k, n - 1)
    d, i = NearestNeighbors(n_neighbors=kk + 1).fit(Z).kneighbors(Z)
    return d[:, 1:], i[:, 1:]

t0 = time.time()
for p, rows in pd.Series(np.arange(len(df))).groupby(pid).indices.items():
    r = np.asarray(rows)
    if len(r) < 3: continue
    Xp = X[r]
    for Z, k, loc, out, rel in [(emb[r], K_APP, loc_app, out_app, out_app_rel),
                                (XYZ[r], K_POS, loc_pos, out_pos, out_pos_rel)]:
        dist, idx = neigh(Z, k)
        nb = Xp[idx]                                  # (n, k, F)
        loc[r] = (Xp - nb.mean(1)) / (nb.std(1) + 1e-3)
        md_ = dist.mean(1)
        out[r] = md_
        rel[r] = md_ / (np.median(md_) + 1e-6)       # local-outlier ratio (LOF-like)
print(f"local features built in {time.time()-t0:.0f}s")

LOCAL = []
for j, c in enumerate(LESION_NUM):
    df[f"{c}_lapp"] = loc_app[:, j]; df[f"{c}_lpos"] = loc_pos[:, j]
    LOCAL += [f"{c}_lapp", f"{c}_lpos"]
for name, v in [("knn_app_dist", out_app), ("knn_app_lof", out_app_rel),
                ("knn_pos_dist", out_pos), ("knn_pos_lof", out_pos_rel)]:
    df[name] = v; LOCAL.append(name)
del loc_app, loc_pos; gc.collect()
print("global:", len(GLOBAL), "| local:", len(LOCAL))

## 4. Train the four feature sets (same LightGBM recipe as Step 1)

In [ ]:
LGB_PARAMS = dict(objective="binary", learning_rate=0.03, num_leaves=63, max_depth=7, min_child_samples=40,
                  feature_fraction=0.6, bagging_fraction=0.8, bagging_freq=1, lambda_l1=0.1, lambda_l2=1.0,
                  n_estimators=300, verbosity=-1)
BASE_F = LESION_NUM + CAT_COLS
FSETS = {"G0_lesion": BASE_F, "G1_global": BASE_F + GLOBAL,
         "G2_local": BASE_F + LOCAL, "G3_global_local": BASE_F + GLOBAL + LOCAL}

def train_oof(feats, name):
    oof_p = np.zeros(len(df)); imps = []
    for f in range(int(folds.max()) + 1):
        tr = np.where(folds != f)[0]; va = np.where(folds == f)[0]
        pred = np.zeros(len(va))
        for b in range(N_BAGS):
            rs = SEED + 100 * f + b
            rng = np.random.default_rng(rs)
            neg = tr[y[tr] == 0]; neg = rng.choice(neg, int(len(neg) * NEG_SAMPLE_RATIO), replace=False)
            sub = np.r_[tr[y[tr] == 1], neg]
            m = lgb.LGBMClassifier(**LGB_PARAMS, random_state=rs).fit(df.iloc[sub][feats], y[sub])
            pred += m.predict_proba(df.iloc[va][feats])[:, 1] / N_BAGS
            imps.append(pd.Series(m.feature_importances_, index=feats))
        oof_p[va] = pred
        print(f"  [{name}] fold {f}: pAUC={pauc_80(y[va], pred):.4f}")
    return oof_p, pd.concat(imps, axis=1).mean(1).sort_values(ascending=False)

IMP = {}
for name, feats in FSETS.items():
    t0 = time.time()
    oof[f"pgb_{name}"], IMP[name] = train_oof(feats, name)
    print(f"== {name}: {len(feats)} feats, OOF pAUC80={pauc_80(y, oof[f'pgb_{name}']):.4f} ({time.time()-t0:.0f}s)\n")

## 5. Final system: LR stack of PG-Boost + image learners

In [ ]:
def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))

def lr_stack(cols):
    Z = np.stack([logit(oof[c].values) for c in cols], 1); pred = np.zeros(len(oof))
    for f in range(int(folds.max()) + 1):
        tr, va = folds != f, folds == f
        sc = StandardScaler().fit(Z[tr])
        pred[va] = LogisticRegression(C=1.0, max_iter=2000).fit(sc.transform(Z[tr]), y[tr]).predict_proba(sc.transform(Z[va]))[:, 1]
    return pred

oof["pgboost_stack"]     = lr_stack(["pgb_G3_global_local", "oof_emb_lr", "oof_cnn"])
oof["global_stack"]      = lr_stack(["pgb_G1_global", "oof_emb_lr", "oof_cnn"])   # fair non-graph counterpart

## 6. Results + paired tests

In [ ]:
MODELS = {"tab_pnorm(step1)": "oof_tab_pnorm", "stack_xgb_all": "stack_xgb_all",
          "G0_lesion": "pgb_G0_lesion", "G1_global": "pgb_G1_global", "G2_local": "pgb_G2_local",
          "G3_global_local": "pgb_G3_global_local", "global_stack": "global_stack", "pgboost_stack": "pgboost_stack"}
boot = PatientBootstrap(pid, n_boot=300, seed=42)
rows = []
for name, col in MODELS.items():
    r = full_report(name, y, oof[col].values, pid)
    mu, sd = fold_stats(y, oof[col].values, folds); lo, hi = boot.ci(y, oof[col].values)
    r.update(pauc80_fold=f"{mu:.4f} ± {sd:.4f}", pauc80_95CI=f"[{lo:.4f}, {hi:.4f}]"); rows.append(r)
res = pd.DataFrame(rows).set_index("model")
cols = ["pauc80", "pauc80_95CI", "pauc80_fold", "auroc", "sens_spec95", "spec_sens95", "top5_recall", "top10_recall"]
print(res[cols].round(4).to_string())

tests = []
for new, base in [("G2_local", "G0_lesion"), ("G2_local", "G1_global"), ("G3_global_local", "G1_global"),
                  ("G3_global_local", "tab_pnorm(step1)"), ("pgboost_stack", "global_stack"),
                  ("pgboost_stack", "tab_pnorm(step1)"), ("pgboost_stack", "stack_xgb_all")]:
    d, (lo, hi), p = boot.paired_test(y, oof[MODELS[new]].values, oof[MODELS[base]].values)
    tests.append(dict(comparison=f"{new} vs {base}", delta_pauc=round(d, 4), ci95=f"[{lo:+.4f}, {hi:+.4f}]", p_value=round(p, 4)))
tests = pd.DataFrame(tests); print(); print(tests.to_string(index=False))

In [ ]:
print("Top-15 features of PG-Boost (G3):")
print(IMP["G3_global_local"].head(15).round(1).to_string())
grp = IMP["G3_global_local"].groupby(lambda c: "local" if c in LOCAL else ("global" if c in GLOBAL else "lesion")).sum()
print("\nImportance share by group:\n", (grp / grp.sum()).round(3).to_string())

In [ ]:
keep = ["isic_id", "patient_id", "fold", "target"] + [c for c in oof.columns if c.startswith("pgb_")] + ["global_stack", "pgboost_stack"]
oof[keep].to_csv(f"{WORK}/oof_step4b.csv", index=False)
res[cols].to_csv(f"{WORK}/results_step4b.csv"); tests.to_csv(f"{WORK}/tests_step4b.csv", index=False)
IMP["G3_global_local"].to_csv(f"{WORK}/importance_pgboost.csv")
print("saved")